# Ingestion Pipeline Notebook

Demonstrates the actual ingestion pipeline used by `scripts/build_index.py` against the real corpus in `data/raw/` (Kazakhstan road traffic rules, road traffic law, KoAP fragments, insurance law). This notebook imports the real functions from `scripts/build_index.py` rather than reimplementing them, so it stays honest as the script evolves.

It writes its own demo output under `notebooks/_ingestion_demo_output/` and never overwrites `data/chunks.jsonl` (the file the running service actually reads) — that file is only ever produced by `scripts/build_index.py` / `make index`, to keep the pipeline reproducible.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from scripts.build_index import (
    load_raw_documents,
    clean_and_normalize,
    chunk_documents_recursive,
    chunk_documents_by_article,
)

## 1. Load raw documents

`.docx` files are parsed with a lightweight custom OOXML reader (`read_docx_text` in `scripts/build_index.py`) that pulls paragraph text straight out of `word/document.xml`, so the project doesn't need a Word/LibreOffice dependency.

In [2]:
docs = load_raw_documents("../data/raw")
for d in docs:
    print(f"{d['source_file']:45s}  {len(d['text']):>8,} chars")

code_of_administrative_offences.docx           1,931,863 chars
insurance_law.docx                              108,673 chars
road_traffic_law.docx                           272,322 chars
road_traffic_rules.docx                         218,585 chars


## 2. Clean and normalize

`clean_and_normalize` strips page-number artifacts ("Page ", "Стр. "), collapses repeated whitespace, and normalizes blank lines. Legal `.docx` exports from `adilet.zan.kz` don't have the header/footer or hyphenation-across-linebreak problems a scanned PDF would, since paragraphs already come out as clean logical units — the main cleanup need here is footnote/reference-marker noise ("Сноска. ... в редакции Закона РК от ..."), which we deliberately keep, since it's often part of the legally relevant text (amendment history) rather than boilerplate.

In [3]:
sample_raw = docs[0]["text"][:400]
sample_clean = clean_and_normalize(docs[0]["text"])[:400]
print("--- RAW ---\n", sample_raw)
print("\n--- CLEANED ---\n", sample_clean)

--- RAW ---
 Об административных правонарушениях
Кодекс Республики Казахстан от 5 июля 2014 года № 235-V ЗРК.
Примечание ИЗПИ!
Для удобства пользования ИЗПИ создано Содержание
СОДЕРЖАНИЕ
Сноска. Оглавление исключено Законом РК от 02.07.2021 № 63-VII (вводится в действие по истечении шестидесяти календарных дней после дня его первого официального опубликования).
Примечание РЦПИ!
Порядок введения в действие наст

--- CLEANED ---
 Об административных правонарушениях
Кодекс Республики Казахстан от 5 июля 2014 года № 235-V ЗРК.
Примечание ИЗПИ!
Для удобства пользования ИЗПИ создано Содержание
СОДЕРЖАНИЕ
Сноска. Оглавление исключено Законом РК от 02.07.2021 № 63-VII (вводится в действие по истечении шестидесяти календарных дней после дня его первого официального опубликования).
Примечание РЦПИ!
Порядок введения в действие наст


## 3. Chunking strategy 1 — `RecursiveCharacterTextSplitter`

Fixed-size character splitting (500 chars, 50 overlap). Domain-agnostic, but on legal text it slices articles mid-sentence — see the ADR for why this isn't the production choice.

In [4]:
recursive_chunks = chunk_documents_recursive(docs, chunk_size=500, chunk_overlap=50)
print(f"RecursiveCharacterTextSplitter produced {len(recursive_chunks):,} chunks")
recursive_chunks[10]

RecursiveCharacterTextSplitter produced 6,911 chunks


{'chunk_id': 'code_of_administrative_offences.docx_rec_10',
 'text': 'Примечание ИЗПИ!\nПо всему тексту;\nслова "законодательные акты", "законодательных актов", "законодательным актом", "законодательными актами", "законодательного акта" заменены соответственно словами "законы", "законов", "законом", "законами", "закона";',
 'source_file': 'code_of_administrative_offences.docx',
 'char_start': 3174,
 'char_end': 3424,
 'document_version': 'prep_v1'}

## 4. Chunking strategy 2 — article-boundary (section-aware) splitter

Custom regex splitter that treats `РАЗДЕЛ N`, `Глава N`, and `Статья N` lines as hard section boundaries, so each statute article stays a single chunk (further split only if it exceeds 1,600 chars). Bare numbered lines (`"1. ..."`) only start a new section when we are *not* already inside a `Статья` block — otherwise every numbered sub-clause of a KoAP article would fragment into its own chunk (an actual bug this project shipped with earlier: the naive version of this regex matched every `"N. text"` line unconditionally and nearly doubled the chunk count by splitting statute bodies into one chunk per sentence).

In [5]:
article_chunks = chunk_documents_by_article(docs)
print(f"Article-boundary splitter produced {len(article_chunks):,} chunks")

from collections import Counter
ids = [c["chunk_id"] for c in article_chunks]
dupes = {k: v for k, v in Counter(ids).items() if v > 1}
print(f"Duplicate chunk_id count: {len(dupes)} (0 expected)")

Article-boundary splitter produced 2,719 chunks
Duplicate chunk_id count: 0 (0 expected)


In [6]:
# A representative chunk with full required metadata: source_file, section_title,
# chunk_id, char_start, char_end, document_version.
sample = next(c for c in article_chunks if c["source_file"] == "code_of_administrative_offences.docx" and "591" in (c.get("section_title") or ""))
sample

{'chunk_id': 'code_of_administrative_offences.docx_sec_707',
 'text': 'Статья 591. Пользование водителем при управлении транспортным средством телефоном либо радиостанцией\n1. Пользование водителем при управлении транспортным средством телефоном либо радиостанцией –\nвлечет штраф в размере пяти месячных расчетных показателей.\n2. Действие, предусмотренное частью первой настоящей статьи, совершенное повторно в течение года после наложения административного взыскания, –\nвлечет штраф в размере десяти месячных расчетных показателей.\nПримечание. Во время управления транспортным средством разрешается пользоваться телефоном либо радиостанцией посредством применения наушников или громкой связи.',
 'source_file': 'code_of_administrative_offences.docx',
 'section_title': 'Статья 591. Пользование водителем при управлении транспортным средством телефоном либо радиостанцией',
 'char_start': 1203865,
 'char_end': 1204486,
 'document_version': 'prep_v1'}

## 5. Strategy comparison

In [7]:
import pandas as pd

def stats(chunks, label):
    lengths = [len(c["text"]) for c in chunks]
    return {
        "strategy": label,
        "chunk_count": len(chunks),
        "avg_chars": round(sum(lengths) / len(lengths), 1),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
    }

pd.DataFrame([
    stats(recursive_chunks, "RecursiveCharacterTextSplitter (500/50)"),
    stats(article_chunks, "Article-boundary (Статья/Глава-aware)"),
])

,strategy,chunk_count,avg_chars,min_chars,max_chars
0,RecursiveCharacterTextSplitter (500/50),6911,370.2,11,499
1,Article-boundary (Статья/Глава-aware),2719,978.2,18,1600


The recursive splitter produces roughly 2x more chunks at a smaller, more uniform size; the article-boundary splitter produces fewer, variable-length chunks that align with real citation units (one chunk per Статья/Пункт in the common case). This is the reason the production index (`scripts/build_index.py::main`) uses `chunk_documents_by_article` — see `docs/adr/001-chunking.md` and the retrieval-strategy ablation in `02_ablation.ipynb` for the recall@5 consequences.

## 6. Persist demo output (idempotency check)

Writes to a scratch path (not `data/chunks.jsonl`) and re-runs the same chunker to confirm the pipeline is deterministic/idempotent, as required.

In [8]:
import json, os

demo_dir = "_ingestion_demo_output"
os.makedirs(demo_dir, exist_ok=True)
demo_path = os.path.join(demo_dir, "chunks_demo.jsonl")
with open(demo_path, "w", encoding="utf-8") as f:
    for c in article_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

rerun = chunk_documents_by_article(docs)
print("Idempotent (same chunk count + same ids on rerun):", [c["chunk_id"] for c in article_chunks] == [c["chunk_id"] for c in rerun])
print(f"Wrote {len(article_chunks):,} chunks to {demo_path}")

Idempotent (same chunk count + same ids on rerun): True
Wrote 2,719 chunks to _ingestion_demo_output/chunks_demo.jsonl
